
# Naive Bayes: Play Tennis Prediction

## Project Introduction

In this project, we use an online Play Tennis dataset to predict whether tennis should be played based on weather conditions.

We use the Universal Data Cleaner to inspect and clean the data before training a Naive Bayes model.



## 1. Install Required Libraries

We install the Universal Data Cleaner package needed for the data-cleaning stage.


In [3]:

%pip install universal-data-cleaner scikit-learn pandas


Note: you may need to restart the kernel to use updated packages.



## 2. Import Required Libraries

We import the libraries used to load data, clean it, encode categories, train the model, and evaluate predictions.


In [4]:

import pandas as pd

from data_cleaner import DataCleaner, CleanerConfig
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score



## 3. Load the Online Play Tennis Dataset

We load the classic Play Tennis CSV file directly from a public GitHub repository.

We display the original dataset before cleaning.


In [5]:

url = "https://raw.githubusercontent.com/PrajwalBharadwajN/VTU_ML_LAB/master/datasets/tennis.csv"

df_original = pd.read_csv(url)

df_original.columns = df_original.columns.str.strip()

print("Original Dataset:")
display(df_original)

print("Dataset Shape:", df_original.shape)

print("\nMissing Values:")
display(df_original.isnull().sum())


Original Dataset:


,Outlook,Temperature,Humidity,Windy,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rainy,Mild,High,Weak,Yes
4,Rainy,Cool,Normal,Weak,Yes
5,Rainy,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rainy,Mild,Normal,Weak,Yes


Dataset Shape: (14, 5)

Missing Values:


Outlook        0
Temperature    0
Humidity       0
Windy          0
PlayTennis     0
dtype: int64


## 4. Prepare the Input Features and Target

We keep the original dataset unchanged.

The weather columns are our input features, and PlayTennis is the target we want to predict.


In [6]:

target_column = next(
    column for column in df_original.columns
    if column.replace("_", "").replace(" ", "").lower() == "playtennis"
)

X_original = df_original.drop(columns=[target_column]).copy()
y = df_original[target_column].copy()

print("Original Input Features:")
display(X_original)

print("Target Values:")
display(y.value_counts())


Original Input Features:


,Outlook,Temperature,Humidity,Windy
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rainy,Mild,High,Weak
4,Rainy,Cool,Normal,Weak
5,Rainy,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rainy,Mild,Normal,Weak


Target Values:


PlayTennis
Yes    9
No     5
Name: count, dtype: int64


## 5. Clean the Dataset Using Universal Data Cleaner

We fit the cleaner on the weather features and transform the same data using its learned cleaning rules.

The original dataset remains unchanged for comparison.


In [8]:

cleaner = DataCleaner(CleanerConfig())

cleaner.fit(X_original)

X_clean = cleaner.transform(X_original)

print("Cleaned Dataset:")
display(X_clean)

print("Cleaned Dataset Shape:", X_clean.shape)

print("\nMissing Values After Cleaning:")
display(X_clean.isnull().sum())

print("\nCleaning Summary:")
display(cleaner.cleaning_summary())


Cleaned Dataset:


,Outlook,Temperature,Humidity,Windy
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rainy,Mild,High,Weak
4,Rainy,Cool,Normal,Weak
5,Rainy,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rainy,Mild,Normal,Weak


Cleaned Dataset Shape: (14, 4)

Missing Values After Cleaning:


Outlook        0
Temperature    0
Humidity       0
Windy          0
dtype: int64


Cleaning Summary:


,column,kind,missing_in_fit,fill_value,lower_bound,upper_bound,outliers_capped,status,generated_columns
0,Outlook,categorical,0,Rainy,NaN,NaN,0,retained,
1,Temperature,categorical,0,Mild,NaN,NaN,0,retained,
2,Humidity,categorical,0,High,NaN,NaN,0,retained,
3,Windy,categorical,0,Weak,NaN,NaN,0,retained,



## 6. Compare Original and Cleaned Data

We compare the original and cleaned weather features to see whether cleaning changed any values, columns, or missing-value counts.


In [9]:

print("Original Data:")
display(X_original)

print("Cleaned Data:")
display(X_clean)

print("Original Shape:", X_original.shape)
print("Cleaned Shape:", X_clean.shape)

comparison = pd.DataFrame({
    "Missing Before": X_original.isnull().sum(),
    "Missing After": X_clean.isnull().sum()
})

display(comparison)

print("Number of Changed Cells:",
      (X_original.astype(str).reset_index(drop=True)
       != X_clean.astype(str).reset_index(drop=True)).sum().sum())


Original Data:


,Outlook,Temperature,Humidity,Windy
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rainy,Mild,High,Weak
4,Rainy,Cool,Normal,Weak
5,Rainy,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rainy,Mild,Normal,Weak


Cleaned Data:


,Outlook,Temperature,Humidity,Windy
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rainy,Mild,High,Weak
4,Rainy,Cool,Normal,Weak
5,Rainy,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rainy,Mild,Normal,Weak


Original Shape: (14, 4)
Cleaned Shape: (14, 4)


,Missing Before,Missing After
Outlook,0,0
Temperature,0,0
Humidity,0,0
Windy,0,0


Number of Changed Cells: 0



## 7. Encode Categorical Values

Categorical Naive Bayes needs categorical inputs in integer-coded form.

We encode the cleaned weather features and the target labels.


In [10]:

encoder = OrdinalEncoder()

X_encoded = encoder.fit_transform(X_clean).astype(int)

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Encoded Features:")
display(pd.DataFrame(X_encoded, columns=X_clean.columns))

print("Encoded Target:")
print(y_encoded)


Encoded Features:


,Outlook,Temperature,Humidity,Windy
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1
5,1,0,1,0
6,0,0,1,0
7,2,2,0,1
8,2,0,1,1
9,1,2,1,1


Encoded Target:
[0 0 1 1 1 0 1 0 1 1 1 1 1 0]



## 8. Train the Naive Bayes Model

We train a Categorical Naive Bayes classifier using the encoded weather data and target labels.


In [11]:

model = CategoricalNB()

model.fit(X_encoded, y_encoded)

print("Model training complete!")


Model training complete!



## 9. Predict Whether to Play Tennis

We provide new weather conditions and ask the trained model to predict Yes or No.



In [ ]:

new_weather = pd.DataFrame({
    column: [X_original[column].mode()[0]]
    for column in X_original.columns
})

new_weather.loc[0, "Outlook"] = "Sunny"
new_weather.loc[0, "Temperature"] = "Cool"
new_weather.loc[0, "Humidity"] = "Normal"

if "Wind" in new_weather.columns:
    new_weather.loc[0, "Wind"] = "Weak"
elif "Windy" in new_weather.columns:
    new_weather.loc[0, "Windy"] = "Weak"

new_weather_clean = cleaner.transform(new_weather)

new_weather_encoded = encoder.transform(new_weather_clean).astype(int)

prediction = model.predict(new_weather_encoded)

result = label_encoder.inverse_transform(prediction)

print("New Weather Conditions:")
display(new_weather)

print("Prediction:", result[0])


New Weather Conditions:


,Outlook,Temperature,Humidity,Windy
0,Sunny,Cool,Normal,Weak


Prediction: Yes



## 10. Evaluate the Model

We calculate training accuracy by comparing predictions on the training dataset with the actual labels.

Training accuracy is not the same as accuracy on unseen data.


In [14]:

y_pred = model.predict(X_encoded)

accuracy = accuracy_score(y_encoded, y_pred)

print(f"Training Accuracy: {accuracy * 100:.2f}%")


Training Accuracy: 92.86%



## 11. Conclusion

In this project, we loaded an online Play Tennis dataset, displayed the original data, applied Universal Data Cleaner, compared the data before and after cleaning, encoded categorical values, trained a Categorical Naive Bayes model, and made a prediction.

This project demonstrates the basic workflow of a supervised machine learning classification task.



# FAAAAAHHHHHHHHHHHHHHHHHHH 🔥
